In [1]:
import pandas as pd
import numpy as np

import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
from IPython.display import display, clear_output

import sys
sys.path.append("../scripts/python/")

In [2]:
df = pd.read_csv("../data/raw/bach_chorales.dataset")
pitch_cols = df.columns[2:14]
df[pitch_cols] = (df[pitch_cols] == "YES") # Converts YES to 1 and NO to 0 (hopefully)

In [3]:
import sounddevice as sd
import threading
import time

from playback import get_audio

current_audio = None
playhead = 0
is_playing = False
sample_rate = 44100

def audio_callback(outdata, frames, time_info, status):
    global playhead, current_audio, is_playing

    if not is_playing or current_audio is None:
        outdata[:] = 0
        return

    end = playhead + frames
    chunk = current_audio[playhead:end]

    if len(chunk) < frames:
        outdata[:len(chunk), 0] = chunk
        outdata[len(chunk):, 0] = 0
        is_playing = False
    else:
        outdata[:, 0] = chunk

    playhead = end

stream = sd.OutputStream(
    samplerate=sample_rate,
    channels=1,
    callback=audio_callback
)
stream.start()

In [4]:
from pitch_utils import inversion, INV_MAP, INV_LABELS

INV_COLORS = [
    "#1f77b4",  # root
    "#ff7f0e",  # 1st
    "#2ca02c",  # 2nd
    "#d62728",  # 3rd
    "#7f7f7f"   # non-chord
]

df["inversion"] = df.apply(
    lambda r: inversion(r["chord_label"], r["bass"]), 
    axis=1
)
df["inv_code"] = df["inversion"].map(INV_MAP)

cmap = ListedColormap(INV_COLORS)

legend_patches = [
    mpatches.Patch(color=INV_COLORS[i], label=INV_LABELS[i])
    for i in range(len(INV_LABELS))
]

In [5]:
def plot_piece(piece_id):
    piece = df[df["choral_ID"] == piece_id]

    fig = plt.figure(figsize=(14, 14))
    gs = gridspec.GridSpec(3, 2, height_ratios=[1.2, 1, 1.2])

    ax0 = fig.add_subplot(gs[0, :])
    ax1 = fig.add_subplot(gs[1, :])
    ax2 = fig.add_subplot(gs[2, 0])
    ax3 = fig.add_subplot(gs[2, 1])

    
    #################################################
    seq = piece["inv_code"].values
    meter = piece["meter"].values
    chords = piece["chord_label"].values
    x = np.arange(len(seq))
    
    meter = meter / np.max(meter)
    
    bars = ax0.bar(x, meter, width=0.9, align="center")
    
    for i, b in enumerate(bars):
        b.set_color(cmap(seq[i] / 4))
    
    ax0.set_xticks(x)
    ax0.set_xticklabels(chords, rotation=45, fontsize=6)
    ax0.set_title("Inversions sequence and importance of event")
    ax0.set_ylim(0, 1)
    ax0.legend(handles=legend_patches, loc="upper right")
    #################################################
    #################################################
    ax1.set_title("dunno yet")
    ax1.axis("off")
    #################################################
    #################################################
    mat = np.zeros((5, 5))
    for i in range(len(seq) - 1):
        mat[seq[i], seq[i+1]] += 1

    row_sums = mat.sum(axis=1, keepdims=True)
    mat = np.divide(mat, row_sums, out=np.zeros_like(mat), where=row_sums != 0)

    im0 = ax2.imshow(mat, cmap="viridis")
    ax2.set_xticks(range(5))
    ax2.set_yticks(range(5))
    ax2.set_xticklabels(INV_LABELS, rotation=45)
    ax2.set_yticklabels(INV_LABELS)
    ax2.set_title("Inversion transitions")
    plt.colorbar(im0, ax=ax2, fraction=0.046, pad=0.04)
    #################################################
    #################################################
    seq = piece["chord_label"].values
    labels = sorted(pd.unique(seq))
    idx = {c: i for i, c in enumerate(labels)}

    mat2 = np.zeros((len(labels), len(labels)))
    for i in range(len(seq) - 1):
        mat2[idx[seq[i]], idx[seq[i+1]]] += 1

    row_sums = mat2.sum(axis=1, keepdims=True)
    mat2 = np.divide(mat2, row_sums, out=np.zeros_like(mat2), where=row_sums != 0)

    im1 = ax3.imshow(mat2, cmap="viridis")
    ax3.set_xticks(range(len(labels)))
    ax3.set_yticks(range(len(labels)))
    ax3.set_xticklabels(labels, rotation=90)
    ax3.set_yticklabels(labels)
    ax3.set_title("Chord transitions")
    plt.colorbar(im1, ax=ax3, fraction=0.046, pad=0.04)
    #################################################
    
    plt.tight_layout()

    with plot_out:
        plot_out.clear_output(wait=True)
        display(fig)
        plt.close(fig)


In [6]:
# Leaderboards
def compute_counts(piece):
    seq = piece["inv_code"].values
    chords = piece["chord_label"].values

    N = len(seq)
    uniq = len(pd.unique(chords))

    non_chord_code = 4
    non_chord_count = int(np.sum(seq == non_chord_code))
    non_chord_ratio = non_chord_count / N if N > 0 else 0.0

    return {
        "n_events": int(N),
        "n_unique_chords": int(uniq),
        "non_chord_count": non_chord_count,
        "non_chord_ratio": float(non_chord_ratio)
    }

counts_df = (
    df.groupby("choral_ID", sort=False)
      .apply(compute_counts)
      .apply(pd.Series)
)

def print_top(df_, col, ascending, k=10, title=""):
    sub = df_.sort_values(col, ascending=ascending).head(k)
    print(title)
    for i, (idx, row) in enumerate(sub.iterrows(), 1):
        print(f"{i:2d}. {idx:15s} {row[col]}")
    print()

rank_out = widgets.Output()
display(rank_out)

def show_all_rankings():
    with rank_out:
        rank_out.clear_output(wait=True)

        print_top(counts_df, "n_events", False, title="Most events")
        print_top(counts_df, "n_events", True,  title="Least events")

        print_top(counts_df, "n_unique_chords", False, title="Most unique chords")
        print_top(counts_df, "n_unique_chords", True,  title="Least unique chords")

        print_top(counts_df, "non_chord_count", False, title="Most non-chord events")
        print_top(counts_df, "non_chord_count", True,  title="Least non-chord events")

        print_top(counts_df, "non_chord_ratio", False, title="Highest non-chord ratio")
        print_top(counts_df, "non_chord_ratio", True,  title="Lowest non-chord ratio")

show_all_rankings()

Output()

In [7]:
# WIDGETS
options = [("Select chorale", None)] + [
    (cid, cid) for cid in df["choral_ID"].unique()
]

dropdown = widgets.Dropdown(
    options=options,
    value=None,
    description="Chorale:"
)

play_btn = widgets.Button(description="Play")
pause_btn = widgets.Button(description="Pause")
restart_btn = widgets.Button(description="Restart")

slider = widgets.IntSlider(description="Position", min=0, max=1, value=0)

# WIDGET'S FUNCTIONS
def load(change):
    global current_audio, playhead
    
    piece_id = change["new"]
    if piece_id is None:
        return
        
    chorale_df = df[df["choral_ID"] == piece_id]
    current_audio = get_audio(chorale_df, piece_id, pitch_cols, "bass", "../data/")
    
    playhead = 0
    slider.max = len(current_audio)

    plot_piece(piece_id)
dropdown.observe(load, names="value")

def play(_):
    global is_playing
    is_playing = True

def pause(_):
    global is_playing
    is_playing = False

def restart(_):
    global playhead
    playhead = 0

play_btn.on_click(play)
pause_btn.on_click(pause)
restart_btn.on_click(restart)

def seek(change):
    global playhead
    playhead = change["new"]
slider.observe(seek, names="value")

def sync():
    global playhead
    while True:
        if current_audio is not None:
            slider.value = min(playhead, slider.max)
        time.sleep(0.1)
threading.Thread(target=sync, daemon=True).start()

In [8]:
display(dropdown)
display(widgets.HBox([play_btn, pause_btn, restart_btn]))
display(slider)

plot_out = widgets.Output() 
display(plot_out)
# keep medium to low volume for less distortion

Dropdown(description='Chorale:', options=(('Select chorale', None), ('000106b_', '000106b_'), ('000206b_', '00…

IntSlider(value=0, description='Position', max=1)

Output()